# tiny-diffusion — overnight training

Pretrain the diffusion LM on TinyStories → finetune on tool trajectories → eval.
Checkpoints go to Google Drive, so a dropped runtime costs you nothing.

**Setup (once):** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all** and walk away.

In the morning, scroll to the last cell's output for the eval numbers.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
cd /content && rm -rf tiny-diffusion && git clone https://github.com/OpenSourceRulesz/tiny-diffusion && cd tiny-diffusion && mkdir -p data /content/drive/MyDrive/tiny-diffusion/checkpoints && ls

In [ ]:
%%bash
cd /content/tiny-diffusion && curl -L -o data/TinyStories-train.txt https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt && ls -lh data/

In [ ]:
%%bash
cd /content/tiny-diffusion
CKPT=/content/drive/MyDrive/tiny-diffusion/checkpoints
echo "=== 1/3 pretraining on TinyStories ==="
PYTHONPATH=src python -m tinydiff.train --data tinystories --data-path data/TinyStories-train.txt --vocab-size 8000 --steps 60000 --batch 64 --d-model 256 --layers 6 --heads 8 --ckpt $CKPT/pre1
echo "=== 2/3 finetuning on tool trajectories ==="
PYTHONPATH=src python -m tinydiff.train --data synthetic --init-ckpt $CKPT/pre1 --tokenizer $CKPT/pre1/tokenizer.json --steps 8000 --n-tool 8000 --hi 9999 --d-model 256 --layers 6 --heads 8 --ckpt $CKPT/agent1
echo "=== 3/3 eval ==="
PYTHONPATH=src python -m tinydiff.eval --ckpt $CKPT/agent1

## Morning after

The eval numbers are in the output of the cell above. The trained agent lives at `/content/drive/MyDrive/tiny-diffusion/checkpoints/agent1`.

If the runtime died mid-run, re-run the notebook with this added to the training cell's first command (everything else identical):

`--resume /content/drive/MyDrive/tiny-diffusion/checkpoints/pre1`

Try it out:

```bash
%%bash
cd /content/tiny-diffusion
PYTHONPATH=src python -m tinydiff.sample --ckpt /content/drive/MyDrive/tiny-diffusion/checkpoints/agent1 --tool-loop --prompt "What is 238*47+12?"
```